# ray.get_actor()

ray.get_actor() 是 Ray 框架中用于获取一个已命名的 Actor 句柄（handle）的核心 API。<font color='red'>它允许你在创建 Actor 的原始作用域之外，通过其唯一的名称来访问和控制这个 Actor，是实现 Actor 跨任务、跨进程通信的关键。</font>

## 🎯 核心功能与目的

在 Ray 中，Actor 默认的生命周期与其创建者（driver 或 job）绑定。当创建它的 job 结束时，Actor 也会被自动删除。<font color='red'>ray.get_actor() 的主要目的就是打破这种限制，</font>让你可以：

### 跨 Job 访问 Actor：在不同的脚本或任务中获取同一个 Actor 的引用。
### 实现有状态服务：创建一个长期运行的、有状态的 Actor，并让多个客户端与之交互。

## 📝 函数签名与参数

In [ ]:
ray.get_actor(name, namespace=None)

- name (str): 这是 Actor 的唯一标识符。你必须先使用 ActorClass.options(name="...") 来创建一个命名的 Actor。
- namespace (str, optional):<font color='red'> 这是 Actor 所在的命名空间。如果未指定（为 None），Ray 会使用当前 driver 的命名空间。命名空间用于隔离不同项目或任务的 Actor，防止命名冲突。</font>

## 🚀 关键使用前提

要成功使用 ray.get_actor()，目标 Actor 在创建时必须满足以下两个条件：
1. 必须被命名：使用 .options(name="your_actor_name") 进行命名。
2. 生命周期必须为 detached：使用 .options(lifetime="detached")。<font color='red'>这使得 Actor 在其创建者 job 结束后依然存活。</font>
      - <font color='red'>如果 Actor 不是 detached 的，即使它被命名了，一旦创建它的 job 结束，ray.get_actor() 也会失败。</font>

## 📖 使用示例

以下示例展示了如何创建一个 detached 的命名 Actor，并在另一个 job 中获取它。
### 创建 Actor (job_1.py)

In [ ]:
import ray

# 初始化 Ray，并指定一个命名空间
ray.init(address="auto", namespace="my_project")

@ray.remote
class Counter:
    def __init__(self):
        self.value = 0
    
    def increment(self):
        self.value += 1
        return self.value

# 创建一个名为 "global_counter" 的 detached Actor
counter = Counter.options(name="global_counter", lifetime="detached").remote()

print("Actor 'global_counter' created in namespace 'my_project'.")
# job_1.py 执行结束，但由于 lifetime="detached"，Actor 仍然存在

### 获取 Actor (job_2.py)

In [ ]:
import ray

# 连接到同一个命名空间
ray.init(address="auto", namespace="my_project")

# 成功获取到在 job_1.py 中创建的 Actor
counter = ray.get_actor("global_counter")

# 现在可以调用它的方法
for _ in range(3):
    # 输出: 1, 2, 3 (状态被保留)
    print(ray.get(counter.increment.remote()))

# 如果尝试从不同的命名空间获取，将会失败
# ray.init(address="auto", namespace="other_project")
# ray.get_actor("global_counter") # 这会抛出异常

### ⚠️ 重要注意事项
- 命名空间隔离：Actor 的名称在命名空间内是唯一的。如上例所示，如果你在不同的命名空间中尝试获取 Actor，必须显式指定其命名空间：ray.get_actor("global_counter", namespace="my_project")。
- <font color='red'>手动清理：detached Actor 不会自动销毁。当你确定不再需要它时，必须手动清理以释放资源。有两种方式：</font>
    - 使用 ray.kill(actor_handle) 强制终止 Actor。
    - 在 Actor 内部调用 ray.actor.exit_actor() 使其自我退出。
- 状态持久性：<font color='red'>Ray 的 Actor 状态存储在内存中。</font>如果承载 Actor 的节点发生故障，Actor 会根据 max_restarts 参数设置进行重启，但其在内存中的状态会丢失，并通过重新运行构造函数来重建。<font color='red'>如果需要真正的状态持久化，你需要自己实现（例如，将状态保存到外部数据库或文件系统中）。</font>